# Weighted Conformal Prediction under Covariate Shift — Extended Benchmark
**22 candidate Kaggle datasets (15 binary + 7 multiclass, loaded via `kagglehub`; any that fail to download are skipped and reported) · 10 methods · 3 base models.**

| Method | Description |
|---|---|
| Unweighted | Standard split conformal (LAC score) |
| Oracle-W | Weighted CP (Tibshirani et al. 2019) with the *true* tilt ratio |
| LR-W | Weighted CP, ratio from a logistic domain classifier |
| KLIEP-W | Weighted CP, ratio from KLIEP (Sugiyama et al. 2008) |
| uLSIF-W | Weighted CP, ratio from uLSIF (Kanamori et al. 2009) |
| RuLSIF-W | Weighted CP, ratio from relative uLSIF, α=0.1 (Yamada et al. 2013); the bounded relative ratio is used directly as the weight |
| nexCP | Non-exchangeable split CP (Barber et al. 2023) with fixed data-independent weights = estimated ratio at calibration points, test weight 1 |
| ESS-Tempered | **Proposed**: LR-W with weights tempered as w^τ, largest τ with ESS ratio ≥ 0.3 |
| ACI | Adaptive conformal inference (Gibbs & Candès 2021), online over the shifted test stream |
| ACI-LR-W | ACI whose base quantile is the LR-weighted conformal quantile |

Set `MODE='fast'` first; use `'full'` for final paper numbers. At the end copy the **RESULTS BLOCK** and paste it back for manuscript drafting.

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import kagglehub
from scipy import stats
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import f_classif
warnings.filterwarnings('ignore')

MODE  = 'fast'      # 'fast' ≈ 15-40 min on Colab CPU (3 seeds) ; 'full' ≈ 1-3 h (10 seeds, use for the paper)
ALPHA = 0.10
CFG = {'fast': dict(N_SEEDS=3,  MAX_N=6000,  GAMMAS=[0.0, 1.0, 2.0],            N_TEST=600),
       'full': dict(N_SEEDS=10, MAX_N=20000, GAMMAS=[0.0, 0.5, 1.0, 1.5, 2.0], N_TEST=1000)}[MODE]
ESS_MIN, ACI_ETA = 0.30, 0.01
OUT = '/content/results'; os.makedirs(OUT, exist_ok=True)

# name: (kaggle handle, target column).  Binary first, then multiclass.
DATASETS = {
 # ---- binary ----
 'adult':      ('uciml/adult-census-income', 'income'),
 'telco':      ('blastchar/telco-customer-churn', 'Churn'),
 'credit':     ('uciml/default-of-credit-card-clients-dataset', 'default.payment.next.month'),
 'heart_fed':  ('fedesoriano/heart-failure-prediction', 'HeartDisease'),
 'heart_uci':  ('johnsmith88/heart-disease-dataset', 'target'),
 'pima':       ('uciml/pima-indians-diabetes-database', 'Outcome'),
 'bcw':        ('uciml/breast-cancer-wisconsin-data', 'diagnosis'),
 'bank':       ('henriqueyamahata/bank-marketing', 'y'),
 'stroke':     ('fedesoriano/stroke-prediction-dataset', 'stroke'),
 'titanic':    ('yasserh/titanic-dataset', 'Survived'),
 'shoppers':   ('henrysue/online-shoppers-intention', 'Revenue'),
 'airline':    ('teejmahal20/airline-passenger-satisfaction', 'satisfaction'),
 'water':      ('adityakadiwal/water-potability', 'Potability'),
 'bankchurn':  ('shrutimechlearn/churn-modelling', 'Exited'),
 'loan':       ('teertha/personal-loan-modeling', 'Personal Loan'),
 # ---- multiclass ----
 'wine':       ('yasserh/wine-quality-dataset', 'quality'),
 'mobile':     ('iabhishekofficial/mobile-price-classification', 'price_range'),
 'obesity':    ('fatemehmehrparvar/obesity-levels', 'NObeyesdad'),
 'fetal':      ('andrewmvd/fetal-health-classification', 'fetal_health'),
 'covertype':  ('uciml/forest-cover-type-dataset', 'Cover_Type'),
 'stellar':    ('fedesoriano/stellar-classification-dataset-sdss17', 'class'),
 'crop':       ('atharvaingle/crop-recommendation-dataset', 'label'),
}
MODELS = {
 'LogReg': lambda: LogisticRegression(max_iter=1000),
 'HGB':    lambda: HistGradientBoostingClassifier(max_iter=100),
 'MLP':    lambda: MLPClassifier((64,), max_iter=200, early_stopping=True, random_state=0),
}

In [ ]:
# ----------------------------------------------------------------- data loading
def read_csv_any(f):
    with open(f, 'r', errors='ignore') as fh: head = fh.readline()
    return pd.read_csv(f, sep=';' if head.count(';') > head.count(',') else ',')

ID_COLS = {'id','customerid','clientnum','rownumber','surname','index','subject'}

def load(handle, target, max_n):
    path = kagglehub.dataset_download(handle)
    csvs = sorted(glob.glob(os.path.join(path, '**', '*.csv'), recursive=True), key=os.path.getsize, reverse=True)
    df = None
    for f in csvs:
        d = read_csv_any(f); d.columns = [str(c).strip() for c in d.columns]
        low = {c.lower(): c for c in d.columns}
        if target in d.columns: df = d; break
        if target.lower() in low: df = d.rename(columns={low[target.lower()]: target}); break
    assert df is not None, f'target {target!r} not found in {handle}'
    df = df.dropna(axis=1, how='all')
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')].replace('?', np.nan)
    df = df.drop(columns=[c for c in df.columns if c.lower() in ID_COLS and c != target])
    for c in df.columns:                                  # numeric-looking strings
        if c != target and df[c].dtype == object:
            conv = pd.to_numeric(df[c], errors='coerce')
            if conv.notna().mean() > 0.95: df[c] = conv
    df = df.drop(columns=[c for c in df.columns if c != target and df[c].isna().mean() > 0.4])
    for c in df.columns:
        if c != target and pd.api.types.is_numeric_dtype(df[c]): df[c] = df[c].fillna(df[c].median())
    df = df.dropna()
    if len(df) > max_n: df = df.sample(max_n, random_state=0)
    codes = pd.factorize(df[target].astype(str).str.strip())[0]
    keep = np.isin(codes, np.where(np.bincount(codes) >= 30)[0])       # drop ultra-rare classes
    df, y = df[keep], pd.factorize(codes[keep])[0]
    X = df.drop(columns=[target])
    X = X.drop(columns=[c for c in X.columns if X[c].dtype == object and X[c].nunique() > 50])
    X = X.loc[:, X.nunique() > 1]
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 10] \
          or [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    F, _ = f_classif(X[num].astype(float).values, y); zcol = num[int(np.argmax(np.nan_to_num(F)))]
    z = X[zcol].astype(float).values; z = np.clip((z - z.mean()) / (z.std() + 1e-9), -3, 3)
    X = pd.get_dummies(X, drop_first=True).astype(float)
    return X.values, y, z, zcol

# ----------------------------------------------------------------- conformal core
def lac_scores(p, y): return 1 - p[np.arange(len(y)), y]

def wquantile(s_cal, w_cal, w_test, alpha):
    """Weighted conformal quantile (Tibshirani et al. 2019); test point has mass at +inf."""
    o = np.argsort(s_cal); s = s_cal[o]; cw = np.cumsum(w_cal[o]); W = cw[-1]
    idx = np.searchsorted(cw, (1 - alpha) * (W + np.asarray(w_test)), side='left')
    return np.where(idx < len(s), s[np.minimum(idx, len(s) - 1)], np.inf)

def evaluate(p_te, y_te, q):
    q = np.broadcast_to(np.asarray(q, dtype=float), (len(y_te),))
    sets = (1 - p_te) <= q[:, None]
    return sets[np.arange(len(y_te)), y_te].mean(), sets.sum(1).mean()

def ess_ratio(w): return (w.sum() ** 2) / (len(w) * (w ** 2).sum())

def temper(wc, wt, rho=ESS_MIN):
    for tau in [1.0, 0.75, 0.5, 0.25, 0.0]:
        if ess_ratio(wc ** tau) >= rho: break
    return wc ** tau, wt ** tau, tau

def aci(p_te, y_te, s_cal, w_cal, w_te, alpha, eta=ACI_ETA):
    """Adaptive conformal inference (Gibbs & Candes 2021) on the test stream, labels revealed online."""
    o = np.argsort(s_cal); s = s_cal[o]; cw = np.cumsum(w_cal[o]); W = cw[-1]; n = len(s)
    a, covs, sizes = alpha, 0, 0
    for t in range(len(y_te)):
        lvl = 1 - a
        if lvl >= 1: q = np.inf
        elif lvl <= 0: q = -np.inf
        else:
            idx = np.searchsorted(cw, lvl * (W + w_te[t]), side='left')
            q = s[idx] if idx < n else np.inf
        st = (1 - p_te[t]) <= q
        hit = st[y_te[t]]; covs += hit; sizes += st.sum()
        a += eta * (alpha - (0.0 if hit else 1.0))
    return covs / len(y_te), sizes / len(y_te)

# ----------------------------------------------------------------- density-ratio estimators
SIGMA_MULTS = [0.1, 0.2, 0.35, 0.6, 1.0, 2.0]   # kernel width = multiple of median pairwise distance (CV-selected)
def sqdist(A, B): return np.maximum((A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T, 0)
def gauss(A, C, s): return np.exp(-sqdist(A, C) / (2 * s * s))

def _setup(Xc, Xt, seed, nc=100):
    rng = np.random.RandomState(seed)
    cidx = rng.choice(len(Xt), min(nc, len(Xt)), replace=False); C = Xt[cidx]
    P = np.vstack([Xc, Xt]); S = P[rng.choice(len(P), min(400, len(P)), replace=False)]
    d = sqdist(S, S)[np.triu_indices(len(S), 1)]
    return C, np.sqrt(np.median(d)) + 1e-9, rng, cidx

def ulsif(Xc, Xt, alpha=0.0, seed=0):
    """(Relative) uLSIF; r_alpha(x)=p_te/(alpha p_te+(1-alpha)p_cal). Sigma/lambda by 5-fold CV."""
    C, med, rng, _ = _setup(Xc, Xt, seed)
    fc, ft = rng.randint(0, 5, len(Xc)), rng.randint(0, 5, len(Xt))
    def HH(Pc, Pt):
        H = (1 - alpha) * Pc.T @ Pc / len(Pc) + alpha * Pt.T @ Pt / len(Pt); return H, Pt.mean(0)
    best = (np.inf, None, None)
    for m in SIGMA_MULTS:
        s = m * med; Pc, Pt = gauss(Xc, C, s), gauss(Xt, C, s)
        for lam in [0.01, 0.1, 1.0]:
            J = 0
            for k in range(5):
                H, h = HH(Pc[fc != k], Pt[ft != k]); th = np.maximum(np.linalg.solve(H + lam * np.eye(len(C)), h), 0)
                Hv, hv = HH(Pc[fc == k], Pt[ft == k]); J += 0.5 * th @ Hv @ th - hv @ th
            if J < best[0]: best = (J, s, lam)
    _, s, lam = best; Pc, Pt = gauss(Xc, C, s), gauss(Xt, C, s)
    H, h = HH(Pc, Pt); th = np.maximum(np.linalg.solve(H + lam * np.eye(len(C)), h), 0)
    fl = lambda r: np.maximum(r, 1e-3 * (np.mean(r) + 1e-12))
    return fl(Pc @ th), fl(Pt @ th)

def _kliep_fit(A, b, iters=150, eps=0.05):
    a = np.ones(A.shape[1]); a /= (b @ a)
    for _ in range(iters):
        a = a + eps * A.T @ (1.0 / np.maximum(A @ a, 1e-12)) / len(A)
        a = a + (1 - b @ a) * b / (b @ b); a = np.maximum(a, 0); a /= (b @ a)
    return a

def kliep(Xc, Xt, seed=0):
    """KLIEP: maximise mean log r(x_te) s.t. mean_cal r = 1; sigma by 3-fold likelihood CV."""
    C, med, rng, cidx = _setup(Xc, Xt, seed); ft = rng.randint(0, 3, len(Xt)); best = (-np.inf, None)
    for m in SIGMA_MULTS:
        s = m * med; Pc, Pt = gauss(Xc, C, s), gauss(Xt, C, s); b = Pc.mean(0); sc = 0
        for k in range(3):
            cols = ft[cidx] != k                      # centres must not come from the held-out fold
            a = _kliep_fit(Pt[ft != k][:, cols], b[cols]); sc += np.mean(np.log(np.maximum(Pt[ft == k][:, cols] @ a, 1e-12)))
        if sc > best[0]: best = (sc, s)
    s = best[1]; Pc, Pt = gauss(Xc, C, s), gauss(Xt, C, s); a = _kliep_fit(Pt, Pc.mean(0))
    fl = lambda r: np.maximum(r, 1e-3 * (np.mean(r) + 1e-12))
    return fl(Pc @ a), fl(Pt @ a)

def lr_ratio(Xc, Xt):
    D = np.vstack([Xc, Xt]); dl = np.r_[np.zeros(len(Xc)), np.ones(len(Xt))]
    p = cross_val_predict(LogisticRegression(max_iter=300), D, dl, cv=5, method='predict_proba')[:, 1]
    w = np.clip(p, 0.01, 0.99); w = w / (1 - w) * len(Xc) / len(Xt); return w[:len(Xc)], w[len(Xc):]

In [ ]:
# ----------------------------------------------------------------- main experiment
METHODS = ['Unweighted','Oracle-W','LR-W','KLIEP-W','uLSIF-W','RuLSIF-W','nexCP','ESS-Tempered','ACI','ACI-LR-W']
rows, meta, failed = [], [], {}
t0 = time.time()
for dname, (h, t) in DATASETS.items():
    try: X, y, z, zcol = load(h, t, CFG['MAX_N'])
    except Exception as e:
        failed[dname] = f'{type(e).__name__}: {str(e)[:100]}'; print(f'SKIP {dname}: {failed[dname]}'); continue
    K = len(np.unique(y)); kind = 'binary' if K == 2 else 'multiclass'
    meta.append(dict(dataset=dname, kind=kind, n=len(y), d=X.shape[1], K=K, shift_feature=zcol))
    for seed in range(CFG['N_SEEDS']):
        rng = np.random.RandomState(seed); n = len(y); idx = rng.permutation(n)
        tr, ca, po = idx[:int(.4*n)], idx[int(.4*n):int(.6*n)], idx[int(.6*n):]
        if len(np.unique(y[tr])) < K: continue
        Xs = StandardScaler().fit(X[tr]).transform(X)
        fitted = {mn: mk().fit(Xs[tr], y[tr]) for mn, mk in MODELS.items()}
        p_cal = {mn: m.predict_proba(Xs[ca]) for mn, m in fitted.items()}
        s_cal = {mn: lac_scores(p_cal[mn], y[ca]) for mn in fitted}
        ntest = min(CFG['N_TEST'], len(po) // 2)
        for g in CFG['GAMMAS']:
            pz = np.exp(g * z[po]); pz /= pz.sum()
            te = rng.choice(po, size=ntest, replace=True, p=pz)   # with replacement => realised shift matches exp(gamma z)
            Xc, Xt = Xs[ca], Xs[te]
            woc, wot = np.exp(g * z[ca]), np.exp(g * z[te])
            wl = lr_ratio(Xc, Xt); wk = kliep(Xc, Xt, seed); wu = ulsif(Xc, Xt, 0.0, seed); wr = ulsif(Xc, Xt, 0.1, seed)
            wtmp = temper(*wl); tau = wtmp[2]
            nrm = lambda w: w / w.mean()
            WS = {'Oracle-W': (woc, wot), 'LR-W': wl, 'KLIEP-W': wk, 'uLSIF-W': wu, 'RuLSIF-W': wr,
                  'ESS-Tempered': wtmp[:2]}
            for mn, m in fitted.items():
                p_te = m.predict_proba(Xs[te]); sc = s_cal[mn]; yt = y[te]
                res = {}
                res['Unweighted'] = (evaluate(p_te, yt, wquantile(sc, np.ones(len(sc)), 1.0, ALPHA)), np.ones(len(sc)))
                for k_, (wc, wt) in WS.items(): res[k_] = (evaluate(p_te, yt, wquantile(sc, wc, wt, ALPHA)), wc)
                wn = nrm(wl[0]); res['nexCP'] = (evaluate(p_te, yt, wquantile(sc, wn, 1.0, ALPHA)), wn)
                res['ACI'] = (aci(p_te, yt, sc, np.ones(len(sc)), np.ones(ntest), ALPHA), np.ones(len(sc)))
                res['ACI-LR-W'] = (aci(p_te, yt, sc, wl[0], wl[1], ALPHA), wl[0])
                for mth, ((cov, size), wc) in res.items():
                    rows.append(dict(dataset=dname, kind=kind, model=mn, seed=seed, gamma=g, method=mth,
                                     coverage=cov, size=size, ess=ess_ratio(wc),
                                     wmae=np.mean(np.abs(nrm(wc) - nrm(woc))),
                                     tau=tau if mth == 'ESS-Tempered' else np.nan))
    print(f'{dname:10s} {kind:10s} done  ({time.time()-t0:.0f}s elapsed)')
R = pd.DataFrame(rows); M = pd.DataFrame(meta)
R.to_csv(f'{OUT}/raw_results.csv', index=False); M.to_csv(f'{OUT}/datasets_loaded.csv', index=False)
print(f'\nLoaded {len(M)} datasets ({(M.kind=="binary").sum()} binary, {(M.kind=="multiclass").sum()} multiclass); failed: {list(failed)}')

In [ ]:
# ----------------------------------------------------------------- analysis
R['gap'] = (R.coverage - (1 - ALPHA)).abs()
R['under'] = (R.coverage < (1 - ALPHA) - 0.02).astype(float)
G = sorted(R.gamma.unique()); gmax = max(G)
# dataset-level means (each dataset counts once -> avoids large datasets dominating)
D = R.groupby(['dataset', 'kind', 'gamma', 'method'])[['coverage', 'gap', 'size', 'under', 'ess', 'wmae']].mean().reset_index()

def pivot(col, d=D):
    return d.groupby(['method', 'gamma'])[col].mean().unstack('gamma').reindex(METHODS).round(3)
def ci95(x):
    x = np.asarray(x); return stats.t.ppf(.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
T_cov, T_gap, T_size, T_under = pivot('coverage'), pivot('gap'), pivot('size'), pivot('under')
T_gap_ci = D[D.gamma == gmax].groupby('method').gap.agg(['mean', ci95]).reindex(METHODS).round(3)
T_kind = D[D.gamma == gmax].groupby(['kind', 'method'])[['coverage', 'gap', 'size']].mean().round(3)
T_model = R[R.gamma == gmax].groupby(['model', 'method'])[['coverage', 'gap', 'size']].mean().round(3)
Dshift = D[D.gamma > 0]
P = Dshift.groupby(['dataset', 'method']).gap.mean().unstack('method')[METHODS]
ranks = P.rank(axis=1).mean().round(2)
try: fr = stats.friedmanchisquare(*[P[m] for m in METHODS])
except Exception: fr = None
wil = {}
for m in METHODS:
    if m != 'ESS-Tempered':
        try: wil[m] = round(stats.wilcoxon(P['ESS-Tempered'], P[m]).pvalue, 4)
        except Exception: wil[m] = np.nan
T_ds = D[D.gamma == gmax].pivot(index='dataset', columns='method', values='gap')[METHODS].round(3)
T_tau = R[R.method == 'ESS-Tempered'].groupby('gamma').tau.mean().round(2)
T_ratio = D[(D.gamma > 0) & D.method.isin(['LR-W','KLIEP-W','uLSIF-W','RuLSIF-W'])].groupby('method')[['wmae','ess','gap']].mean().round(3)
for n_, T in [('cov', T_cov), ('gap', T_gap), ('size', T_size), ('under', T_under), ('dataset_gap', T_ds)]:
    T.to_csv(f'{OUT}/table_{n_}.csv'); open(f'{OUT}/table_{n_}.tex', 'w').write(T.to_latex())

fig, ax = plt.subplots(1, 4, figsize=(21, 4))
for mth in METHODS:
    d = D[D.method == mth].groupby('gamma'); nd = D.dataset.nunique()
    for a_, col in zip(ax[:3], ['coverage', 'gap', 'size']):
        a_.errorbar(d[col].mean().index, d[col].mean(), d[col].std() / np.sqrt(nd), marker='o', ms=3, label=mth,
                    ls='--' if mth.startswith('ACI') else '-')
ax[0].axhline(1 - ALPHA, c='k', ls=':'); ax[0].set_title('Coverage (target 0.90)'); ax[1].set_title('|coverage − 0.90|'); ax[2].set_title('Average set size')
for a_ in ax[:3]: a_.set_xlabel('shift strength γ')
ax[0].legend(fontsize=6)
for mth, c in zip(['LR-W','KLIEP-W','uLSIF-W','RuLSIF-W'], ['C0','C1','C2','C3']):
    d = D[(D.method == mth) & (D.gamma > 0)]; ax[3].scatter(d.wmae, d.gap, s=8, alpha=.5, c=c, label=mth)
ax[3].set_xlabel('weight error (normalised MAE vs oracle)'); ax[3].set_ylabel('|coverage − 0.90|'); ax[3].set_title('Coverage gap vs ratio-estimation error'); ax[3].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_main.pdf'); plt.savefig(f'{OUT}/fig_main.png', dpi=150); plt.show()

print('=========== RESULTS BLOCK (copy everything between the markers) ===========')
print(f'MODE={MODE} alpha={ALPHA} cfg={CFG} ESS_MIN={ESS_MIN} ACI_eta={ACI_ETA}')
print('\nDatasets loaded:\n', M.to_string(index=False)); print('\nFailed:', failed)
for nm, T in [('Coverage', T_cov), ('|Coverage-0.9|', T_gap), ('Avg set size', T_size), ('Under-coverage rate (<0.88)', T_under)]:
    print(f'\n{nm}  (rows: method, cols: gamma; dataset-level means)\n', T.to_string())
print(f'\nGap at gamma={gmax}, mean and 95% CI half-width across datasets:\n', T_gap_ci.to_string())
print('\nBy dataset kind at gamma_max:\n', T_kind.to_string())
print('\nBy base model at gamma_max:\n', T_model.to_string())
print('\nAverage rank of |gap| across datasets (gamma>0; lower=better):\n', ranks.to_string())
print('\nFriedman test:', fr)
print('Wilcoxon p-values, ESS-Tempered vs method (paired over datasets, gamma>0):', wil)
print('\nMean tau selected by ESS-Tempered per gamma:\n', T_tau.to_string())
print('\nDensity-ratio estimator quality (gamma>0):\n', T_ratio.to_string())
print(f'\nPer-dataset |gap| at gamma={gmax}:\n', T_ds.to_string())
print('=========== END RESULTS BLOCK ===========')

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files; files.download('/content/results.zip')